# ViHSD Mixture of Experts experiment

This Colab entry point mounts Google Drive, installs the project dependencies, and runs the root-level training and evaluation scripts. Edit `configs/vihsd.yaml` before starting if the dataset schema or hyperparameters need to change.

## 1. Mount Google Drive

The YAML checkpoint path points to `/content/drive/MyDrive/ViHSD-MoE/checkpoints`. Drive must be mounted before training so `.safetensors` files persist after the Colab runtime ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone the GitHub repository and install dependencies

This notebook treats GitHub as the source of truth. Each runtime clones the latest `main` branch into `/content/moe-vihsd`, installs dependencies from that clone, and runs the scripts there.

In [ ]:
PROJECT_DIR = '/content/moe-vihsd'
REPOSITORY_URL = 'https://github.com/lngphgthao/moe-vihsd.git'

!rm -rf $PROJECT_DIR
!git clone --depth 1 --branch main $REPOSITORY_URL $PROJECT_DIR
%cd $PROJECT_DIR
%pip install -q -r requirements.txt

In [ ]:
import os
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Create a Colab Secret named HF_TOKEN before continuing.')
os.environ['HF_TOKEN'] = hf_token
os.environ['CHECKPOINT_DIR'] = '/content/drive/MyDrive/ViHSD-MoE/checkpoints'
os.environ['RESULTS_DIR'] = f'{PROJECT_DIR}/results'
print('Hugging Face authentication configured from Colab Secrets.')

## 4. Train the full-parameter MoE

All embeddings, attention layers, router parameters, expert parameters, and classifier parameters are optimized. The best validation checkpoint is saved to Google Drive under a timestamped run folder.

In [ ]:
!CHECKPOINT_DIR=$CHECKPOINT_DIR RESULTS_DIR=$RESULTS_DIR python train.py --config configs/vihsd.yaml --no-smoke-test

## 5. Evaluate the newest run and save JSON predictions

This loads the latest checkpoint from Drive, evaluates the test split, records expert routing counts, and writes results into the cloned repository.

In [ ]:
!CHECKPOINT_DIR=$CHECKPOINT_DIR RESULTS_DIR=$RESULTS_DIR python evaluate.py --config configs/vihsd.yaml